In [1]:
import os
import h5py
import pandas as pd
from tqdm import tqdm
import numpy as np

In [25]:
from diffusers import StableDiffusionPipeline, UNet2DConditionModel, DDIMScheduler
from datasets import load_dataset
import torch
from torch.utils.data import Dataset, DataLoader

In [20]:
def load_spectrograms(df, h5_path, class_label):
    class_indices = df[df['primary_label'] == class_label].index.values
    
    with h5py.File(h5_path, 'r') as src_h5:
        src_dset = src_h5['spectrograms']
        spectrograms = src_dset[class_indices]
    
    spectrograms = (spectrograms - spectrograms.min(axis=(1, 2), keepdims=True)) / (
        spectrograms.max(axis=(1, 2), keepdims=True) - spectrograms.min(axis=(1, 2), keepdims=True)
    )
    return spectrograms

In [7]:
class SpectrogramDataset(Dataset):
    def __init__(self, spectrograms):
        self.spectrograms = spectrograms

    def __len__(self):
        return len(self.spectrograms)

    def __getitem__(self, idx):
        spectrogram = self.spectrograms[idx]
        spectrogram = torch.tensor(spectrogram, dtype=torch.float32).unsqueeze(0)  # shape: (1, 256, 256)
        return spectrogram

In [11]:
h5_path = 'processed_data/expanded_spectrograms.h5'

In [13]:
original_df = pd.read_csv('processed_data/expanded_df_formatted.csv')
spec_averages = np.load('processed_data/spectrogram_averages.npy')

In [18]:
original_df['primary_label'].value_counts().sort_values(ascending=True)

primary_label
21116        2
42113        3
42087        3
66016        4
868458       4
          ... 
trokin    2887
whtdov    2927
roahaw    3171
grekis    3479
compau    3664
Name: count, Length: 206, dtype: int64

In [22]:
spectrograms = load_spectrograms(original_df, h5_path, class_label='21116')
dataset = SpectrogramDataset(spectrograms)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

In [26]:
unet = UNet2DConditionModel.from_pretrained("stabilityai/stable-diffusion-2", subfolder="unet")

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
unet.to(device)

Cannot initialize model with low cpu memory usage because `accelerate` was not found in the environment. Defaulting to `low_cpu_mem_usage=False`. It is strongly recommended to install `accelerate` for faster and less memory-intense model loading. You can do so with: 
```
pip install accelerate
```
.


diffusion_pytorch_model.safetensors:   2%|2         | 73.4M/3.46G [00:00<?, ?B/s]

UNet2DConditionModel(
  (conv_in): Conv2d(4, 320, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (time_proj): Timesteps()
  (time_embedding): TimestepEmbedding(
    (linear_1): Linear(in_features=320, out_features=1280, bias=True)
    (act): SiLU()
    (linear_2): Linear(in_features=1280, out_features=1280, bias=True)
  )
  (down_blocks): ModuleList(
    (0): CrossAttnDownBlock2D(
      (attentions): ModuleList(
        (0-1): 2 x Transformer2DModel(
          (norm): GroupNorm(32, 320, eps=1e-06, affine=True)
          (proj_in): Linear(in_features=320, out_features=320, bias=True)
          (transformer_blocks): ModuleList(
            (0): BasicTransformerBlock(
              (norm1): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
              (attn1): Attention(
                (to_q): Linear(in_features=320, out_features=320, bias=False)
                (to_k): Linear(in_features=320, out_features=320, bias=False)
                (to_v): Linear(in_features=320, out_f

In [33]:
optimizer = torch.optim.AdamW(unet.parameters(), lr=1e-4)
scheduler = DDIMScheduler.from_pretrained("stabilityai/stable-diffusion-2", subfolder="scheduler")

num_epochs = 10
for epoch in range(num_epochs):
    unet.train()
    for batch in tqdm(dataloader):
        batch = batch.to(device)
        
        noise = torch.randn_like(batch)
        timesteps = torch.randint(0, scheduler.num_train_timesteps, (batch.size(0),), device=device).long()

        noisy_batch = scheduler.add_noise(batch, noise, timesteps)
        noisy_batch = noisy_batch.repeat(1, 4, 1, 1)

        dummy_encoder_hidden_states = torch.randn(batch.size(0), 77, 1024, device=device)

        noise_pred = unet(noisy_batch, timesteps, encoder_hidden_states=dummy_encoder_hidden_states).sample
        
        loss = torch.nn.functional.mse_loss(noise_pred, noise)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {loss.item()}")

  0%|                                                                                                                                         | 0/1 [03:25<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 2.00 MiB. GPU 0 has a total capacity of 8.00 GiB of which 0 bytes is free. Including non-PyTorch memory, this process has 17179869184.00 GiB memory in use. Of the allocated memory 22.00 GiB is allocated by PyTorch, and 151.61 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)